# Chapter 11 — Problem Set 2: Solutions

Runnable solutions for advanced LLM problems.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))
import numpy as np
np.random.seed(42)

## 1. Top-k Sampling — Solution

In [ ]:
def softmax(x):
    x = x - x.max()
    e = np.exp(x); return e / e.sum()

def my_top_k(logits, k, rng):
    top_idx = np.argpartition(-logits, k - 1)[:k]
    probs = softmax(logits[top_idx])
    return int(top_idx[rng.choice(len(top_idx), p=probs)])

# Compare empirical distributions
from generation_utils import top_k_sample
logits = np.array([3.0, 2.5, 1.0, 0.5, -0.5, -1.0, -2.0])
k = 3
n = 5000
rng_a = np.random.default_rng(0)
rng_b = np.random.default_rng(0)
counts_mine = np.zeros_like(logits)
counts_ref  = np.zeros_like(logits)
for _ in range(n):
    counts_mine[my_top_k(logits, k, rng_a)] += 1
    counts_ref[top_k_sample(logits, k=k, rng=rng_b)] += 1
print('mine:', (counts_mine / n).round(3))
print('ref :', (counts_ref / n).round(3))

## 2. Tiny Transformer Block — Solution

In [ ]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True); sd = x.std(-1, keepdims=True)
    return (x - mu) / (sd + eps)

class TinyBlock:
    def __init__(self, d_model, ffn_hidden, seed=0):
        rng = np.random.default_rng(seed); s = 1.0 / np.sqrt(d_model)
        self.Wq = rng.standard_normal((d_model, d_model)) * s
        self.Wk = rng.standard_normal((d_model, d_model)) * s
        self.Wv = rng.standard_normal((d_model, d_model)) * s
        self.W1 = rng.standard_normal((d_model, ffn_hidden)) * s
        self.W2 = rng.standard_normal((ffn_hidden, d_model)) * s

    def attn(self, x):
        Q, K, V = x @ self.Wq, x @ self.Wk, x @ self.Wv
        scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(Q.shape[-1])
        scores -= scores.max(-1, keepdims=True); w = np.exp(scores); w /= w.sum(-1, keepdims=True)
        return w @ V

    def __call__(self, x):
        x = x + self.attn(layer_norm(x))               # pre-norm
        x = x + np.maximum(0, layer_norm(x) @ self.W1) @ self.W2
        return x

block = TinyBlock(d_model=16, ffn_hidden=32)
x = np.random.randn(1, 5, 16)
y = block(x)
print('input :', x.shape)
print('output:', y.shape)

## 3. Perplexity — Solution

In [ ]:
from generation_utils import perplexity

log_probs_A = [-1.20, -0.85, -0.50, -1.10, -0.95]
log_probs_B = [-2.30, -1.95, -2.10, -1.80, -2.50]

ppl_A = perplexity(log_probs_A)
ppl_B = perplexity(log_probs_B)
print(f'PPL_A = {ppl_A:.3f}')
print(f'PPL_B = {ppl_B:.3f}')
print('Lower is better -> Model A fits the held-out text much better.')

## 4. Embedding-Based Classifier — Solution

In [ ]:
from llm_utils import EmbeddingExtractor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

texts = [
    'Cats and dogs are popular pets.',          'Pets like cats are common.',
    'Hamsters and rabbits are also popular pets.',
    'Stocks rose on the announcement.',         'Equities jumped after the report.',
    'Bond yields climbed after the central bank statement.',
    'The team won the championship.',           'They scored in overtime to clinch the title.',
    'The striker scored twice to win the cup final.',
    'Pizza and pasta are Italian classics.',    'Italian food includes pizza, pasta and gelato.',
    'Lasagne and risotto are classic Italian dishes.',
]
labels = [0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3]

extractor = EmbeddingExtractor(dim=64)
X = extractor.embed(texts)
clf = LogisticRegression(max_iter=500, random_state=42)
# 3-fold CV: 4 classes x 3 samples each gives 8 train / 4 test per fold
scores = cross_val_score(clf, X, labels, cv=3)
print('CV accuracy per fold:', scores.round(3))
print('Mean:', scores.mean().round(3))

## 5. Prompt vs Context Window — Solution

**Two strategies that fit in 4 096 tokens:**

- **Truncate**: send the first ~3 800 tokens and the question. Cheap; loses tail content.
- **Map-reduce / chunked summarisation**: split the doc into K chunks of ~800 tokens, summarise each, then ask the question over the summaries. Costs K calls but preserves coverage.

**Trade-off**: fewer-but-longer chunks preserve local context (good for narrative, code), more-but-shorter chunks improve recall but lose coherence and may double-count overlap.

**Retrieval**: with embedding-based retrieval (Chapter 13 RAG) you only send the chunks that match the query — typically 3–5 of them — keeping prompts small and answer quality high. This is almost always the right answer for QA over long documents.

## 6. Evaluate Generations — Solution

In [ ]:
GENS = ['paris is the capital of france', 'the sun is a star', 'water boils at 100 c']
REFS = ['paris', 'the sun is a star at the centre', 'water boils at 100 degrees celsius']

def exact_match(g, r):
    return float(g.strip().lower() == r.strip().lower())

def bow_f1(g, r):
    gs, rs = set(g.lower().split()), set(r.lower().split())
    if not gs or not rs:
        return 0.0
    p = len(gs & rs) / len(gs)
    rcl = len(gs & rs) / len(rs)
    return 0.0 if (p + rcl) == 0 else 2 * p * rcl / (p + rcl)

em_scores = [exact_match(g, r) for g, r in zip(GENS, REFS)]
f1_scores = [bow_f1(g, r) for g, r in zip(GENS, REFS)]
print('Exact-match accuracy:', np.mean(em_scores))
print('Mean BoW F1:', round(np.mean(f1_scores), 3))

print('\nWhy neither is sufficient:')
print('- Exact match punishes paraphrase ("paris" vs "paris is the capital of france").')
print('- Bag-of-words F1 ignores word order and meaning ("not good" == "good not").')
print('- Use win-rate, embedding-based similarity, or LLM-as-judge for open-ended generation.')